# 00 — Data Pipeline

## Purpose
This is the **first notebook that must run**. It does pure ETL (Extract, Transform, Load) — no analysis, no graphs.

It takes the raw inputs and produces the two intermediate files that every other notebook depends on:

| Input | Output |
|-------|--------|
| 186+ raw POS CSV files (`data/raw/`) | `data/intermediate/df_complete_186_files.csv` — all branches concatenated |
| `data/weather/Clima Monterrey.csv` | `data/weather/Clima_limpio.csv` — cleaned weather (date + tavg only) |
| Both of the above | `data/intermediate/complete data with weather.csv` — sales merged with temperature |
| The merged file above | `data/intermediate/datanomodifier.csv` — modifier rows removed, `day_part` and `cold_or_warm` added |

## Run order
**Run this notebook first.** All other notebooks read files that this one produces.

## Libraries needed
`pip install pandas`

## Step 1 — Imports and path setup

In [ ]:
import pandas as pd
import glob
import os
import numpy as np

# Path to the project root (one level up from notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR         = os.path.join(PROJECT_ROOT, "data", "raw")
WEATHER_DIR     = os.path.join(PROJECT_ROOT, "data", "weather")
INTERMEDIATE_DIR= os.path.join(PROJECT_ROOT, "data", "intermediate")

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)

## Step 2 — Load all 186 raw CSV files and concatenate

The raw data comes from the POS (point-of-sale) system exported as individual CSV files,
one file per branch per date range. We load all of them at once using `glob` and stack them
into a single dataframe.

**Why `low_memory=False`?** Some columns have mixed types across files (e.g., `is_modifier`
appears as `True/False` strings in some files and booleans in others). This flag prevents
pandas from making premature type guesses.

In [ ]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(csv_files)} CSV files")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df["_source_file"] = os.path.basename(f)  # track which file each row came from
        dfs.append(df)
    except Exception as e:
        print(f"  ERROR reading {os.path.basename(f)}: {e}")

df_all = pd.concat(dfs, ignore_index=True, sort=False)
print(f"Total rows after concat: {len(df_all):,}")
print(f"Columns: {df_all.columns.tolist()}")

## Step 3 — Fix datetime columns

The POS exports dates as strings. We convert the three timestamp columns to proper
`datetime64` so downstream operations (merges, rolling windows, hour extraction) work correctly.

`errors='coerce'` means any unparseable value becomes `NaT` (Not a Time) instead of crashing.

In [ ]:
datetime_cols = ["operating_date", "closing_time", "captured_time"]
for col in datetime_cols:
    if col in df_all.columns:
        df_all[col] = pd.to_datetime(df_all[col], errors="coerce")

print("Date column types:")
print(df_all[datetime_cols].dtypes)

## Step 4 — Save the concatenated raw file

We save the concatenated raw data before any transformation. This file is the
source of truth for all raw analysis and is also the input for the weather merge.

In [ ]:
output_path = os.path.join(INTERMEDIATE_DIR, "df_complete_186_files.csv")
df_all.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved {len(df_all):,} rows to df_complete_186_files.csv")

## Step 5 — Clean weather data

The Monterrey weather station provides daily observations with many columns
(temperature, precipitation, wind speed, pressure, sunshine duration).

For our forecasting model we only need **`tavg`** (average daily temperature in °C)
because it's the most complete and most directly correlated with food demand.

We:
1. Keep only `date` and `tavg`
2. Normalize the date to day level (remove any time component)
3. Remove duplicate dates (keep one reading per day)
4. Save as `Clima_limpio.csv`

In [ ]:
clima_raw_path = os.path.join(WEATHER_DIR, "Clima Monterrey.csv")
df_clima = pd.read_csv(clima_raw_path)
print("Raw weather columns:", df_clima.columns.tolist())
print(f"Raw weather rows: {len(df_clima)}")
df_clima.head()

In [ ]:
# Keep only the columns we need
df_clima = df_clima[["date", "tavg"]].copy()

# Convert types
df_clima["date"] = pd.to_datetime(df_clima["date"], errors="coerce").dt.normalize()
df_clima["tavg"] = pd.to_numeric(df_clima["tavg"], errors="coerce")

# Remove duplicates — keep first reading if multiple entries per day
df_clima = df_clima.drop_duplicates(subset=["date"])

print(f"Clean weather rows: {len(df_clima)}")
print(df_clima.dtypes)
df_clima.head()

In [ ]:
clima_clean_path = os.path.join(WEATHER_DIR, "Clima_limpio.csv")
df_clima.to_csv(clima_clean_path, index=False, encoding="utf-8")
print(f"Saved Clima_limpio.csv with {len(df_clima)} rows")

## Step 6 — Merge sales data with weather

We join `tavg` onto every sales row by matching `operating_date` = `date`.

**Join type: left merge** — we keep all sales rows even if there's no weather data
for that day (those rows will have `tavg = NaN`).

**`validate='m:1'`** — ensures the weather table has at most one row per date
(it would crash if there were duplicates, which protects us from silent data errors).

In [ ]:
# Reload clean files to avoid any state issues
df_sales = pd.read_csv(os.path.join(INTERMEDIATE_DIR, "df_complete_186_files.csv"), low_memory=False)
df_weather = pd.read_csv(clima_clean_path)

df_sales["operating_date"] = pd.to_datetime(df_sales["operating_date"], errors="coerce").dt.normalize()
df_weather["date"] = pd.to_datetime(df_weather["date"], errors="coerce").dt.normalize()
df_weather["tavg"] = pd.to_numeric(df_weather["tavg"], errors="coerce")

df_merged = df_sales.merge(
    df_weather[["date", "tavg"]],
    left_on="operating_date",
    right_on="date",
    how="left",
    validate="m:1"
).drop(columns=["date"])

print(f"Sales rows:         {len(df_sales):,}")
print(f"Merged rows:        {len(df_merged):,}  (should be identical to sales)")
print(f"Rows missing tavg:  {df_merged['tavg'].isna().sum():,}")

In [ ]:
merged_path = os.path.join(INTERMEDIATE_DIR, "complete data with weather.csv")
df_merged.to_csv(merged_path, index=False, encoding="utf-8")
print(f"Saved complete data with weather.csv with {len(df_merged):,} rows")

## Step 7 — Remove modifier rows and add derived columns

### What are modifiers?
When a customer customizes a product (e.g., 'LECHE DESLACTOSADA' instead of regular milk),
the POS system records the base item on one row AND each customization on a separate row
marked `is_modifier=True`.

For demand forecasting we only care about **how many base products were sold**, not the
customizations. So we remove modifier rows.

### Derived columns added here
- **`cold_or_warm`** — binary label: `'warm'` if `tavg >= 25°C`, else `'cold'`.
  The 25°C threshold was chosen because it roughly matches the seasonal divide in Monterrey
  between comfortable weather and hot weather where customers visibly shift product preferences.
- **`day_part`** — time-of-day segment based on `captured_time`:
  - `morning` = 6:00–11:59
  - `afternoon` = 12:00–18:59
  - `night` = 19:00–5:59

In [ ]:
df = pd.read_csv(merged_path, low_memory=False)

# Parse datetime columns
for col in ["operating_date", "closing_time", "captured_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Fix is_modifier: it arrives as a mixed-type string column ('True'/'False'/NaN)
# We standardize it to a proper boolean
df["is_modifier"] = (
    df["is_modifier"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
    .astype(bool)
)

# cold_or_warm: 25°C is the warm threshold for Monterrey
df["tavg"] = pd.to_numeric(df["tavg"], errors="coerce")
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

# day_part: segment the day into morning / afternoon / night
hour = df["captured_time"].dt.hour
df["day_part"] = np.select(
    [hour.between(6, 11, inclusive="both"),
     hour.between(12, 18, inclusive="both")],
    ["morning", "afternoon"],
    default="night"
)

# Keep only non-modifier rows
df_nomod = df[df["is_modifier"] == False].copy()

print(f"Total rows (with modifiers):    {len(df):,}")
print(f"Rows after removing modifiers:  {len(df_nomod):,}")
print(f"Modifier rows removed:          {len(df) - len(df_nomod):,}")

In [ ]:
nomod_path = os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv")
df_nomod.to_csv(nomod_path, index=False, encoding="utf-8")
print(f"Saved datanomodifier.csv with {len(df_nomod):,} rows")

## Summary

| File created | Location | Description |
|---|---|---|
| `df_complete_186_files.csv` | `data/intermediate/` | All 186 raw CSVs concatenated |
| `Clima_limpio.csv` | `data/weather/` | Weather cleaned to date + tavg |
| `complete data with weather.csv` | `data/intermediate/` | Sales + weather merged |
| `datanomodifier.csv` | `data/intermediate/` | Sales without modifier rows, with `cold_or_warm` and `day_part` |

**Next step:** Run `01_data_cleaning.ipynb`